# ML-07 — Baseline Action Score and Top-10 Review

**Lane: CTR / Engagement Opportunity Scoring**

This notebook builds a transparent, hand-written rule baseline that scores content pages by how far their CTR falls below their position tier's expected CTR — weighted by impression volume so that high-traffic gaps rank higher. The rule is what a Week-5 model must beat.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
import json, os, pathlib

# ── Load data ──────────────────────────────────────────────────────────
DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"
OUT_DIR   = pathlib.Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")

Loaded 30,000 rows × 44 columns


## 1. My rule and its reason codes

### Signal checks — two signals, one bucket table each

Before encoding the rule, I check the two signals it leans on:

1. **CTR-vs-position** (flag-linked: this is the signal behind FlyRank's CTR-fix logic — pages that under-perform their position tier's expected CTR)
2. **Staleness** (flag-linked: this is behind FlyRank's refresh flags — stale pages are more likely to be declining)

Each gets a bucket table with `n` printed and a one-word verdict.

In [2]:
# ── SIGNAL CHECK 1: CTR varies by position tier ──────────────────────
# Premise: expected CTR drops with worse ranking position. If this isn't
# real, comparing a page's CTR to its tier's median is meaningless.
#
# Gotcha: avg_position == 0 means "no data" (1,205 rows) — exclude them.
# The position_tier column silently lumps these into "top_3".

has_pos = df[df["avg_position"] > 0].copy()
print(f"Signal 1: CTR by position tier (n = {len(has_pos):,} rows with valid position)")
print(f"Excluded: {(df['avg_position'] == 0).sum():,} rows with avg_position = 0 (no data)\n")

# Volume filter: only pages with >= 500 impressions so CTR isn't noise
visible = has_pos[has_pos["impressions_90d"] >= 500].copy()
print(f"Volume-filtered (impressions_90d >= 500): n = {len(visible):,}\n")

tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
tier_table = visible.groupby("position_tier")["ctr"].agg(
    n="count", mean="mean", median="median", std="std"
).reindex(tier_order)

print("Bucket table — CTR by position tier (volume-filtered):")
print(tier_table.to_string())
print()
print("Median CTR drops monotonically from top_3 (0.20) → page_1 (0.24) → ")
print("striking (0.17) → page_3_5 (0.09) → deep (0.00).")
print("(top_3 median is below page_1 because top_3 is a small, noisy bucket.)")
print()
print("Verdict: CONFIRMED")
print("CTR differs meaningfully across position tiers. The gap is real and")
print("large enough to build a tier-adjusted score on.")

Signal 1: CTR by position tier (n = 28,795 rows with valid position)
Excluded: 1,205 rows with avg_position = 0 (no data)

Volume-filtered (impressions_90d >= 500): n = 16,726

Bucket table — CTR by position tier (volume-filtered):
                  n      mean  median       std
position_tier                                  
top_3           458  0.346572    0.20  0.421533
page_1         7064  0.338808    0.24  0.350915
striking       4485  0.266798    0.17  0.316403
page_3_5       4330  0.143236    0.09  0.184625
deep            389  0.043213    0.00  0.139540

Median CTR drops monotonically from top_3 (0.20) → page_1 (0.24) → 
striking (0.17) → page_3_5 (0.09) → deep (0.00).
(top_3 median is below page_1 because top_3 is a small, noisy bucket.)

Verdict: CONFIRMED
CTR differs meaningfully across position tiers. The gap is real and
large enough to build a tier-adjusted score on.


In [3]:
# ── SIGNAL CHECK 2: Staleness vs. declining ─────────────────────────
# Premise: pages that haven't been updated recently are more likely to
# be declining. This is behind FlyRank's refresh flags.
#
# Using is_declining only as a label/outcome here — NOT as a feature.

df["is_declining"] = (df["trend_direction"] == "down").astype(int)
base_rate = df["is_declining"].mean()
print(f"Signal 2: Staleness vs. decline (n = {len(df):,}, base decline rate = {base_rate:.3f})\n")

staleness_table = df.groupby("freshness_tier").agg(
    n=("is_declining", "count"),
    decline_rate=("is_declining", "mean")
).reindex(["0-30", "31-90", "91-180", "181+"])

print("Bucket table — decline rate by freshness tier:")
print(staleness_table.to_string())
print()
print(f"Base rate: {base_rate:.3f}")
print("0-30 days since update:  0.511 (below base rate)")
print("91-180 days:             0.611 (above base rate — staler content declines more)")
print("181+:                    0.471 (small n = 174, unreliable)")
print()
print("Verdict: MIXED")
print("The 91-180 bucket (n=9,171) shows a real lift in decline rate above")
print("base rate (0.611 vs 0.542). But the relationship isn't monotonic —")
print("the 181+ bucket reverses, and 68% of the data sits in the 0-30 bucket.")
print("Signal is real for mid-range staleness but not a clean gradient.")
print("This means staleness is useful as a BONUS factor, not the primary score.")

Signal 2: Staleness vs. decline (n = 30,000, base decline rate = 0.542)

Bucket table — decline rate by freshness tier:
                    n  decline_rate
freshness_tier                     
0-30            20480      0.511377
31-90             175      0.588571
91-180           9171      0.611057
181+              174      0.471264

Base rate: 0.542
0-30 days since update:  0.511 (below base rate)
91-180 days:             0.611 (above base rate — staler content declines more)
181+:                    0.471 (small n = 174, unreliable)

Verdict: MIXED
The 91-180 bucket (n=9,171) shows a real lift in decline rate above
base rate (0.611 vs 0.542). But the relationship isn't monotonic —
the 181+ bucket reverses, and 68% of the data sits in the 0-30 bucket.
Signal is real for mid-range staleness but not a clean gradient.
This means staleness is useful as a BONUS factor, not the primary score.


### The rule — in plain words

**A page is worth reviewing for a title/meta/snippet rewrite if:**
1. It has a valid ranking position (`avg_position > 0`) and enough impressions (`≥ 500`) — so the CTR comparison isn't noise.
2. Its CTR is *below* its position tier's median CTR — meaning it's under-capturing clicks for its ranking.
3. The score equals the **CTR gap × impressions** — larger gaps on higher-volume pages rank higher, because fixing them recovers more clicks.
4. A **staleness bonus** of 1.25× applies if the page hasn't been updated in ≥ 91 days — stale pages are likelier to have outdated titles/snippets.

**Reason codes:**
- `ctr_below_tier` — the page's CTR is below its position tier's median

**Action label:**
- `review_title_meta` — review this page's title, meta description, and snippet for a rewrite

This rule is transparent: a non-engineer can read it. No fitted weights, no hidden logic.

## 2. Build the ranked queue (writes the CSV)

Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.

In [4]:
# ── Build the scored queue ─────────────────────────────────────────────

# Start from pages with valid position data
scored = df[df["avg_position"] > 0].copy()

# Step 1: Compute expected (median) CTR per position tier
# Only use volume-filtered pages to compute the tier medians, so low-volume
# noise doesn't drag down the reference CTR.
tier_median = (
    scored[scored["impressions_90d"] >= 500]
    .groupby("position_tier")["ctr"]
    .median()
    .rename("expected_ctr")
)
print("Expected CTR per tier (median from volume-filtered pages):")
print(tier_median.to_string())
print()

# Map expected CTR back to every row
scored = scored.merge(tier_median, left_on="position_tier", right_index=True, how="left")

# Step 2: Compute the CTR gap
scored["ctr_gap"] = scored["expected_ctr"] - scored["ctr"]

# Step 3: Filter to eligible pages
# - Must have enough impressions (>= 500) for a meaningful CTR
# - CTR gap must be positive (below tier median)
eligible = scored[
    (scored["impressions_90d"] >= 500) &
    (scored["ctr_gap"] > 0)
].copy()
print(f"Eligible pages (impressions >= 500, CTR below tier median): {len(eligible):,}")

# Step 4: Score = CTR gap × impressions (higher = bigger opportunity)
eligible["score"] = eligible["ctr_gap"] * eligible["impressions_90d"]

# Step 5: Staleness bonus — 1.25× for pages not updated in >= 91 days
eligible["stale"] = (eligible["days_since_last_update"] >= 91).astype(int)
eligible["score"] = eligible["score"] * (1.0 + 0.25 * eligible["stale"])

# Step 6: Assign reason code and action label
eligible["reason_code"] = "ctr_below_tier"
eligible["action"]      = "review_title_meta"

# Step 7: Rank by score descending
eligible = eligible.sort_values("score", ascending=False).reset_index(drop=True)
eligible["rank"] = range(1, len(eligible) + 1)

# Step 8: Select output columns and write CSV
out_cols = [
    "rank", "content_id", "client_id", "score", "reason_code", "action",
    "ctr", "expected_ctr", "ctr_gap", "avg_position", "position_tier",
    "impressions_90d", "days_since_last_update", "stale"
]
out_df = eligible[out_cols].copy()

csv_path = OUT_DIR / "baseline_action_score.csv"
out_df.to_csv(csv_path, index=False)
print(f"\nWrote {len(out_df):,} rows to {csv_path}")
print(f"\nTop 5 preview:")
print(out_df.head(5).to_string(index=False))

Expected CTR per tier (median from volume-filtered pages):
position_tier
deep        0.00
page_1      0.24
page_3_5    0.09
striking    0.17
top_3       0.20

Eligible pages (impressions >= 500, CTR below tier median): 7,950

Wrote 7,950 rows to ..\outputs\baseline_action_score.csv

Top 5 preview:
 rank           content_id         client_id      score    reason_code            action  ctr  expected_ctr  ctr_gap  avg_position position_tier  impressions_90d  days_since_last_update  stale
    1 content_36ff89c8214e client_19581e27de 70085.5375 ctr_below_tier review_title_meta 0.05          0.24     0.19           7.3        page_1           295097                     104      1
    2 content_5fe46e04994d client_4e07408562 64714.3750 ctr_below_tier review_title_meta 0.14          0.24     0.10           4.2        page_1           517715                     104      1
    3 content_c8e9d6ab9013 client_19581e27de 62603.4000 ctr_below_tier review_title_meta 0.00          0.24     0.24      

In [5]:
# ── Quick sanity metrics ────────────────────────────────────────────
print("Score distribution:")
print(out_df["score"].describe().to_string())
print(f"\nPosition tier distribution in queue:")
print(out_df["position_tier"].value_counts().to_string())
print(f"\nStaleness bonus applied: {out_df['stale'].sum():,} / {len(out_df):,} "
      f"({100*out_df['stale'].mean():.1f}%)")

Score distribution:
count     7950.000000
mean       855.282856
std       2588.809957
min          5.400000
25%        106.922500
50%        234.488750
75%        636.800000
max      70085.537500

Position tier distribution in queue:
position_tier
page_1      3505
striking    2155
page_3_5    2070
top_3        220

Staleness bonus applied: 3,227 / 7,950 (40.6%)


## 3. Top-10 review

For each of the top 10: the action, why it's there, and what would make it wrong.

In [6]:
# ── Top-10 display ────────────────────────────────────────────────────
top10 = out_df.head(10).copy()
print("Top 10 ranked pages:\n")
display_cols = ["rank", "content_id", "score", "action", "reason_code",
                "ctr", "expected_ctr", "ctr_gap", "avg_position",
                "position_tier", "impressions_90d", "days_since_last_update", "stale"]
print(top10[display_cols].to_string(index=False))

Top 10 ranked pages:

 rank           content_id      score            action    reason_code  ctr  expected_ctr  ctr_gap  avg_position position_tier  impressions_90d  days_since_last_update  stale
    1 content_36ff89c8214e 70085.5375 review_title_meta ctr_below_tier 0.05          0.24     0.19           7.3        page_1           295097                     104      1
    2 content_5fe46e04994d 64714.3750 review_title_meta ctr_below_tier 0.14          0.24     0.10           4.2        page_1           517715                     104      1
    3 content_c8e9d6ab9013 62603.4000 review_title_meta ctr_below_tier 0.00          0.24     0.24           9.7        page_1           208678                     104      1
    4 content_c84a0ab98e90 46886.9100 review_title_meta ctr_below_tier 0.03          0.24     0.21           7.8        page_1           223271                      20      0
    5 content_8451fc6f034d 46264.4800 review_title_meta ctr_below_tier 0.03          0.20     0.17     

In [7]:
# ── Top-10 one-line review ─────────────────────────────────────────
# For each: action, why it's there, what would make it wrong.

reviews = []
for _, row in top10.iterrows():
    r = int(row["rank"])
    cid = row["content_id"]
    scr = row["score"]
    ctr_val = row["ctr"]
    exp_ctr = row["expected_ctr"]
    gap = row["ctr_gap"]
    pos = row["avg_position"]
    tier = row["position_tier"]
    imp = int(row["impressions_90d"])
    stale_flag = "yes" if row["stale"] == 1 else "no"
    days_update = int(row["days_since_last_update"])

    # Why it's there
    why = (f"CTR {ctr_val:.2f}% is {gap:.2f}pp below {tier} tier median "
           f"({exp_ctr:.2f}%), with {imp:,} impressions")
    if stale_flag == "yes":
        why += f" + stale ({days_update}d since update, 1.25× boost)"

    # What would make it wrong
    wrong_reasons = []
    if imp > 50000:
        wrong_reasons.append("very high impressions could be brand/navigational — CTR expectations differ")
    if pos <= 3:
        wrong_reasons.append("top-3 position: low CTR might be a SERP feature (featured snippet absorbs clicks)")
    if ctr_val == 0:
        wrong_reasons.append("zero CTR with high impressions could be a measurement artifact or non-clickable SERP feature")
    if gap < 0.10:
        wrong_reasons.append("gap is small — could be random noise rather than a real opportunity")
    if not wrong_reasons:
        wrong_reasons.append("if the page's low CTR is correct for its intent (e.g. informational SERP where users get the answer without clicking), title/meta rewrite won't help")

    review_line = (
        f"#{r} {cid}\n"
        f"   Action: review_title_meta\n"
        f"   Why: {why}\n"
        f"   Wrong if: {'; '.join(wrong_reasons)}"
    )
    reviews.append(review_line)

print("Top-10 Review — one line each:\n")
print("\n\n".join(reviews))

Top-10 Review — one line each:

#1 content_36ff89c8214e
   Action: review_title_meta
   Why: CTR 0.05% is 0.19pp below page_1 tier median (0.24%), with 295,097 impressions + stale (104d since update, 1.25× boost)
   Wrong if: very high impressions could be brand/navigational — CTR expectations differ

#2 content_5fe46e04994d
   Action: review_title_meta
   Why: CTR 0.14% is 0.10pp below page_1 tier median (0.24%), with 517,715 impressions + stale (104d since update, 1.25× boost)
   Wrong if: very high impressions could be brand/navigational — CTR expectations differ; gap is small — could be random noise rather than a real opportunity

#3 content_c8e9d6ab9013
   Action: review_title_meta
   Why: CTR 0.00% is 0.24pp below page_1 tier median (0.24%), with 208,678 impressions + stale (104d since update, 1.25× boost)
   Wrong if: very high impressions could be brand/navigational — CTR expectations differ; zero CTR with high impressions could be a measurement artifact or non-clickable SERP f

## 4. Weak picks + leakage check

Which picks look wrong and why? Confirm no product flags or future windows leaked in.

In [8]:
# ── Weak picks analysis ───────────────────────────────────────────────
print("=== Weak-pick patterns in the top 20 ===")
top20 = out_df.head(20)

# 1. Any zero-CTR pages in top 20? Those are suspicious.
zero_ctr = top20[top20["ctr"] == 0.0]
print(f"\nZero-CTR pages in top 20: {len(zero_ctr)}")
if len(zero_ctr) > 0:
    print("  These are weak picks — zero CTR with high impressions could be")
    print("  measurement artifacts or non-clickable SERP features, not real")
    print("  title/meta opportunities.")
    print(zero_ctr[["rank", "content_id", "ctr", "impressions_90d", "position_tier"]].to_string(index=False))

# 2. Top-3 position pages — might be SERP-feature dominated
top3_pos = top20[top20["position_tier"] == "top_3"]
print(f"\nTop-3 position pages in top 20: {len(top3_pos)}")
if len(top3_pos) > 0:
    print("  Weak: a top-3 page with low CTR likely has a SERP feature")
    print("  (featured snippet, knowledge panel) absorbing clicks — a title")
    print("  rewrite won't fix that.")

# 3. Very high impression pages — might be brand/navigational
high_imp = top20[top20["impressions_90d"] > 50000]
print(f"\nVery high impression (>50k) pages in top 20: {len(high_imp)}")
if len(high_imp) > 0:
    print("  These score high mostly because of volume, not gap size.")
    print("  A brand/navigational page naturally has different CTR behavior.")

=== Weak-pick patterns in the top 20 ===

Zero-CTR pages in top 20: 1
  These are weak picks — zero CTR with high impressions could be
  measurement artifacts or non-clickable SERP features, not real
  title/meta opportunities.
 rank           content_id  ctr  impressions_90d position_tier
    3 content_c8e9d6ab9013  0.0           208678        page_1

Top-3 position pages in top 20: 3
  Weak: a top-3 page with low CTR likely has a SERP feature
  (featured snippet, knowledge panel) absorbing clicks — a title
  rewrite won't fix that.

Very high impression (>50k) pages in top 20: 20
  These score high mostly because of volume, not gap size.
  A brand/navigational page naturally has different CTR behavior.


In [9]:
# ── Leakage check ─────────────────────────────────────────────────────
print("=== Leakage check ===")
print()

# 1. No label-derived inputs
# The label is is_declining, derived from trend_direction / trend_pct.
# Check that NONE of these appear in our scoring columns.
score_inputs = ["ctr", "avg_position", "position_tier", "impressions_90d",
                "days_since_last_update"]
label_cols = {"trend_direction", "trend_pct", "is_declining", "is_declining_label"}
leaked = set(score_inputs) & label_cols
print(f"Label columns used as score inputs: {leaked if leaked else 'NONE ✓'}")

# 2. No product flags (health_score, etc.) used
product_flag_cols = {"health_score", "needs_attention", "quick_win"}
existing_product = product_flag_cols & set(df.columns)
used_product = set(score_inputs) & product_flag_cols
print(f"Product flag columns in dataset: {existing_product if existing_product else 'NONE (not in this CSV)'}")
print(f"Product flag columns used as score inputs: {used_product if used_product else 'NONE ✓'}")

# 3. No future-window columns
# All our inputs use trailing-90-day data; no forward-looking columns.
print(f"\nInputs to the score: {score_inputs}")
print("All are trailing-90-day or static content properties.")
print("No future-window or forward-looking columns used. ✓")
print()
print("Leakage check: PASSED")

=== Leakage check ===

Label columns used as score inputs: NONE ✓
Product flag columns in dataset: NONE (not in this CSV)
Product flag columns used as score inputs: NONE ✓

Inputs to the score: ['ctr', 'avg_position', 'position_tier', 'impressions_90d', 'days_since_last_update']
All are trailing-90-day or static content properties.
No future-window or forward-looking columns used. ✓

Leakage check: PASSED


In [10]:
# ── Save metrics JSON (committable receipt) ────────────────────────
metrics = {
    "notebook": "w04_baseline_score",
    "n_total_rows": len(df),
    "n_valid_position": int((df["avg_position"] > 0).sum()),
    "n_eligible": len(out_df),
    "signal_1_verdict": "CONFIRMED",
    "signal_1_name": "ctr_varies_by_position_tier",
    "signal_2_verdict": "MIXED",
    "signal_2_name": "staleness_vs_declining",
    "rule_description": "score = (expected_ctr - actual_ctr) * impressions_90d * staleness_bonus",
    "reason_code": "ctr_below_tier",
    "action_label": "review_title_meta",
    "score_max": float(out_df["score"].max()),
    "score_median": float(out_df["score"].median()),
    "staleness_bonus_pct": float(100 * out_df["stale"].mean()),
    "leakage_check": "PASSED"
}

json_path = OUT_DIR / "w04_baseline_metrics.json"
with open(json_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {json_path}")
print(json.dumps(metrics, indent=2))

Metrics saved to ..\outputs\w04_baseline_metrics.json
{
  "notebook": "w04_baseline_score",
  "n_total_rows": 30000,
  "n_valid_position": 28795,
  "n_eligible": 7950,
  "signal_1_verdict": "CONFIRMED",
  "signal_1_name": "ctr_varies_by_position_tier",
  "signal_2_verdict": "MIXED",
  "signal_2_name": "staleness_vs_declining",
  "rule_description": "score = (expected_ctr - actual_ctr) * impressions_90d * staleness_bonus",
  "reason_code": "ctr_below_tier",
  "action_label": "review_title_meta",
  "score_max": 70085.5375,
  "score_median": 234.48874999999998,
  "staleness_bonus_pct": 40.59119496855346,
  "leakage_check": "PASSED"
}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Two signal verdicts with visible bucket tables and n (CTR-vs-position: CONFIRMED; staleness: MIXED)
- [x] One rule with a score, a reason code (`ctr_below_tier`), and an action label (`review_title_meta`)
- [x] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`
- [x] Top-10 reviewed with "what would make it wrong" for each
- [x] No future-window or label-derived inputs (leakage check PASSED)
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.